# Setup and Imports

In [27]:
import os
import shutil
import random
import pandas as pd
import tqdm
import numpy as np
import cv2

# Selecting and Copying Images

In [3]:
# Configuration
selected_classes = [0, 1, 14, 17, 38, 13]
num_images_per_class = 100
train_csv_path = "../data/Train.csv"  # relative to /utils/
data_folder_path = "../data/"
selected_folder_path = "../data/selected/"

In [28]:
def extract_selected_images(train_csv_path, data_folder_path, selected_folder_path, selected_classes,  num_images_per_class=100):
    df = pd.read_csv(train_csv_path)

    os.makedirs(selected_folder_path, exist_ok=True)

    for class_id in tqdm.tqdm_notebook(selected_classes):
        # Filtering rows for this class
        class_rows = df[df['ClassId'] == class_id]

        # Randomlying select images
        selected_rows = class_rows.sample(n=num_images_per_class, random_state=42)

        # Creating class folder in selected/
        class_folder = os.path.join(selected_folder_path, str(class_id))
        os.makedirs(class_folder, exist_ok=True)

        for _, row in selected_rows.iterrows():
            relative_path = row['Path']  

            # # For, testing
            # print(relative_path)
            # print()

            src_path = os.path.join(data_folder_path, relative_path) # We take only filename
            dst_path = os.path.join(class_folder, os.path.basename(relative_path))

            # # For, testing
            # print(src_path)
            # print(dst_path)
            # print()

            # break

            # Copying file
            shutil.copy(src_path, dst_path)

    print(f"\nExtracted {num_images_per_class} images for each selected class into {selected_folder_path}")

In [29]:
extract_selected_images(train_csv_path, data_folder_path, selected_folder_path, selected_classes,  num_images_per_class)

C:\Users\user\AppData\Local\Temp\ipykernel_2892\3059438259.py:6: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for class_id in tqdm.tqdm_notebook(selected_classes):


  0%|          | 0/6 [00:00<?, ?it/s]


Extracted 100 images for each selected class into ../data/selected/


# Creating a CSV File for the Selected Images

In [4]:
selected_folder_path = "../data/selected/"
output_csv_path = "../data/selected.csv"

In [20]:
def create_selected_csv(selected_folder_path, output_csv_path, selected_classes, train_csv_path):
    train_df = pd.read_csv(train_csv_path)

    train_df['Filename'] = train_df['Path'].apply(lambda x: os.path.basename(x))

    records = []

    for class_id in tqdm.tqdm_notebook(selected_classes):
        class_folder = os.path.join(selected_folder_path, str(class_id))

        if not os.path.exists(class_folder):
            print(f"\nWarning: Folder {class_folder} does not exist.")

            continue

        for filename in os.listdir(class_folder):
           
            if filename.endswith(".png"):
                relative_path = f"{class_id}/{filename}"  

                # # For, testing
                # print(relative_path)

                # Finding the matching row from train_df
                
                # print(str(train_df['Path'])[6:]) # For, testing

                matching_row = train_df[train_df['Filename'] == filename]

                # print(matching_row) # For, testing

                # break

                if not matching_row.empty:
                    width = matching_row.iloc[0]['Width']
                    height = matching_row.iloc[0]['Height']
                    roi_x1 = matching_row.iloc[0]['Roi.X1']
                    roi_y1 = matching_row.iloc[0]['Roi.Y1']
                    roi_x2 = matching_row.iloc[0]['Roi.X2']
                    roi_y2 = matching_row.iloc[0]['Roi.Y2']
                else:
                    width = height = roi_x1 = roi_y1 = roi_x2 = roi_y2 = None
                    
                    print(f"Warning: No matching entry found for {relative_path} in Train.csv")

                records.append((relative_path, class_id, width, height, roi_x1, roi_y1, roi_x2, roi_y2))

    df = pd.DataFrame(records, columns=[
        'Path', 'ClassId', 'Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2'
    ])
    
    df.to_csv(output_csv_path, index=False)

    print(f"\nCreated selected.csv at {output_csv_path}")

In [21]:
create_selected_csv(selected_folder_path, output_csv_path, selected_classes, train_csv_path)

C:\Users\user\AppData\Local\Temp\ipykernel_17248\4285656807.py:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for class_id in tqdm.tqdm_notebook(selected_classes):


  0%|          | 0/6 [00:00<?, ?it/s]


Created selected.csv at ../data/selected.csv


### EDA: For, testing

In [22]:
selected_classes_data = pd.read_csv(output_csv_path)

In [23]:
selected_classes_data.head()

,Path,ClassId,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2
0,0/00000_00000_00002.png,0,30,30,5,5,25,25
1,0/00000_00000_00005.png,0,31,31,6,6,26,26
2,0/00000_00000_00009.png,0,36,36,6,5,30,30
3,0/00000_00000_00012.png,0,37,38,5,6,32,33
4,0/00000_00000_00015.png,0,44,44,6,6,39,39


In [24]:
selected_classes_data["ClassId"].unique()

array([ 0,  1, 14, 17, 38, 13], dtype=int64)

In [25]:
selected_classes_data["ClassId"].value_counts()

ClassId
0     100
1     100
14    100
17    100
38    100
13    100
Name: count, dtype: int64

# Calculating Meta-data for the Selected Classes

In [28]:
meta_folder_path = "../data/Meta/"

In [30]:
def calculate_features(image_path):
    features = {}
    
    img = cv2.imread(image_path)
    
    if img is None:
        return None

    # Converting to HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hue = hsv[:, :, 0]
    features['avg_hue'] = np.mean(hue)

    # Grayscale and threshold
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Finding contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)

        # Circularity
        if perimeter != 0:
            circ = 4 * np.pi * area / (perimeter ** 2)
        else:
            circ = 0
        features['circ'] = circ

        # Bounding rectangle
        x, y, w, h = cv2.boundingRect(cnt)
        ar = float(w) / h
        rect_area = w * h
        extent = float(area) / rect_area if rect_area != 0 else 0
        features['ar'] = ar
        features['extent'] = extent
    else:
        features['circ'] = features['ar'] = features['extent'] = 0

    # Corners
    corners = cv2.goodFeaturesToTrack(gray, maxCorners=100, qualityLevel=0.01, minDistance=10)
    features['corners'] = 0 if corners is None else len(corners)

    # Masked Color
    mask = thresh
    mean_color = cv2.mean(img, mask=mask.astype(np.uint8))
    features['mask_color'] = mean_color[:3]  # Ignore alpha if present

    return features

def process_images(folder_path):
    data = []
    allowed_labels = {'0', '1', '14', '17', '38', '13'}  # Set for faster lookup

    for filename in os.listdir(folder_path):
        
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            label = os.path.splitext(filename)[0]  # filename without extension

            if label in allowed_labels:
                image_path = os.path.join(folder_path, filename)
                features = calculate_features(image_path)

                if features is not None:
                    record = {
                        'label': label,
                        'avg_hue': features['avg_hue'],
                        'circ': features['circ'],
                        'ar': features['ar'],
                        'extent': features['extent'],
                        'corners': features['corners'],
                        'mask_color_R': features['mask_color'][2],  # OpenCV is BGR
                        'mask_color_G': features['mask_color'][1],
                        'mask_color_B': features['mask_color'][0],
                    }
                    data.append(record)

    df = pd.DataFrame(data)
    df.to_csv('../data/meta.csv', index=False)

    print('\nmeta.csv created successfully.')

In [31]:
process_images(meta_folder_path)


meta.csv created successfully.


### EDA: For, testing

In [32]:
meta_folder_data = pd.read_csv('../data/meta.csv')

In [33]:
meta_folder_data.head()

,label,avg_hue,circ,ar,extent,corners,mask_color_R,mask_color_G,mask_color_B
0,0,2.535600,0.759070,1.000000,0.777100,17,247.630091,242.160933,241.859233
1,1,2.535600,0.759070,1.000000,0.777100,17,247.696245,242.229249,241.927668
2,13,0.970225,0.654840,1.123596,0.606573,16,248.076842,240.993134,240.618960
3,14,1.324400,0.933651,1.000000,0.841250,26,244.035044,232.082603,231.435962
4,17,1.420200,0.512504,1.000000,0.772600,18,245.382302,238.181127,237.765641
